# Multi-Class ANN Classifier (Keras): Penguin Species

Architecture: `Dense(32, relu)` -> `Dense(16, relu)` -> `Dense(n_classes, softmax)`

Before running: upload `penguins_size.csv` to Colab (left sidebar, Files). The target is `species` (Adelie, Chinstrap, Gentoo). `penguins_lter.csv` is not needed.

## 1. Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report

tf.random.set_seed(42)
np.random.seed(42)

## 2. Load the dataset

In [ ]:
# Penguins dataset: use penguins_size.csv (not penguins_lter.csv)
matches = glob.glob('/content/*penguins_size*.csv')
assert matches, 'penguins_size.csv not found in /content. Upload it first.'
DATA_PATH = matches[0]
TARGET_COL = 'species'
print('Using file:', DATA_PATH)

df = pd.read_csv(DATA_PATH)
df['sex'] = df['sex'].replace('.', np.nan)   # one stray '.' value in the sex column
print('Target column:', TARGET_COL)
print('Shape:', df.shape)
df.head()

## 3. Preprocess

- Drop missing rows
- One-hot encode any text feature columns
- Label-encode the target (works for text labels and also remaps numeric labels to 0..n-1, which sparse categorical crossentropy needs)
- Stratified train/test split, then scale features



In [ ]:
df = df.dropna().reset_index(drop=True)

X = df.drop(columns=[TARGET_COL])
y_raw = df[TARGET_COL]

# Features: one-hot encode text columns
X = pd.get_dummies(X, drop_first=True).astype('float32')

# Target: label encode (text -> integers 0..n_classes-1)
le = LabelEncoder()
y = le.fit_transform(y_raw)
class_names = [str(c) for c in le.classes_]
n_classes = len(class_names)
print('Classes:', class_names)
print('n_classes:', n_classes)
print(pd.Series(y_raw).value_counts())

# Split (stratify if every class has at least 2 samples)
try:
    X_train, X_test, y_train, y_test = train_test_split(
        X.values, y, test_size=0.2, random_state=42, stratify=y)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(
        X.values, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print('Train:', X_train.shape, ' Test:', X_test.shape)

## 4. Build and compile the ANN

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(n_classes, activation='softmax'),
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 5. Train (30 epochs, 20% of the training data held out for validation)

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.2,
    verbose=1,
)

## 6. Test accuracy

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f'Test accuracy: {test_acc:.4f}')
print(f'Test loss:     {test_loss:.4f}')

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print()
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

## 7. Training vs validation curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].plot(history.history['accuracy'], label='Train accuracy')
axes[0].plot(history.history['val_accuracy'], label='Validation accuracy')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history['loss'], label='Train loss')
axes[1].plot(history.history['val_loss'], label='Validation loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Why Softmax?

This is a multi-class problem where each sample belongs to exactly one class, so the output layer must turn the raw scores into a probability distribution over all classes that sums to 1. Softmax does exactly that (the predicted class is the one with the highest probability), and it pairs correctly with sparse categorical crossentropy, whereas sigmoid would score each class independently and would not give a single coherent choice.